# Extracting the MJO with Multivariate EOFs: A Simple RMM-like Index

## Preface

In the previous notebook 'spectral_analysis', we introduced spectral analysis using simple time series.

We examined Fourier spectra, Welch spectra, red-noise references, 30–90 day filtering, lead-lag relationships, and longitude-time Hovmöller diagrams.

That notebook showed an important limitation:

> A simple area-mean OLR spectrum can reveal subseasonal variability, but it is not optimized to isolate the Madden–Julian Oscillation.

The MJO is not only a time-scale signal.

It is also a large-scale, eastward-propagating, coupled convection-circulation pattern.

Therefore, a better diagnostic should use both the time scale and the spatial structure of the MJO.

This notebook follows the basic idea of Wheeler and Hendon (2004): use multivariate EOF analysis of equatorial OLR, U850, and U200 to extract a pair of principal components that behave like an MJO index.

## Notebook overview

This notebook has four main goals.

First, we will briefly prepare the same daily tropical datasets used in the previous notebook:

- outgoing longwave radiation, or OLR
- 850-hPa zonal wind, or U850
- 200-hPa zonal wind, or U200

The data preparation will be brief because the previous notebook already explained the details.

Second, we will build a combined multivariate dataset using equatorial latitude-mean OLR, U850, and U200.

Each variable will be normalized before EOF analysis so that no single variable dominates only because of its units or variance.

Third, we will compute EOFs of the combined field.

The leading EOF pair will be interpreted as a simple RMM-like representation of the MJO.

Fourth, we will perform spectral analysis on the resulting principal components.

This is the key difference from the previous notebook.

Instead of asking whether a raw area-mean OLR time series has a strong MJO peak, we ask whether the EOF-extracted PCs concentrate variance in the 30–80 or 30–90 day MJO band.

## Main workflow

The workflow is:

OLR, U850, U200  
→ minimal preprocessing  
→ 15°S–15°N latitude mean  
→ remove seasonal cycle and low-frequency variability  
→ normalize each field  
→ combine variables into one matrix  
→ multivariate EOF analysis  
→ PC1 and PC2  
→ spectral analysis of PC1 and PC2  
→ compare OLR-only, OLR+U850, and OLR+U850+U200

The central idea is:

> Multivariate EOF projection acts as a spatial filter for the MJO.

This is why the spectra of EOF principal components can show a clearer MJO time scale than the spectrum of a simple area-mean OLR time series.

## Connection to Wheeler and Hendon (2004)

Wheeler and Hendon (2004) developed the Real-time Multivariate MJO index, or RMM index.

Their index is based on EOF analysis of combined equatorial OLR, U850, and U200 fields.

A key result of their paper is that the leading pair of PCs has most of its variance concentrated at intraseasonal MJO periods, especially around 30–80 days.

This notebook does not attempt to exactly reproduce the operational RMM index.

Instead, it builds a simplified RMM-like workflow for teaching purposes.

The goal is to understand why multivariate EOFs can extract the MJO more clearly than simple area-mean spectral analysis.

# Part I. Why use multivariate EOFs for MJO analysis?

## 1. Why area-mean spectra are not enough

In the previous notebook, we computed spectra from area-mean OLR, U850, and U200 time series.

That was a useful first step because it showed that tropical convection and winds contain substantial subseasonal variability.

However, the spectra did not show a sharp, isolated MJO peak.

This is not surprising.

The MJO is a propagating pattern, not a local oscillation at one fixed longitude.

At one time, enhanced convection may be located over the Indian Ocean.

A few days or weeks later, it may move toward the Maritime Continent or the western Pacific.

If we average over a broad longitude range, different phases of the MJO can partially cancel each other.

As a result, a simple area-mean time series may contain MJO variability, but it is not the best way to isolate it.

## 2. The MJO has coupled convection and circulation structure

The MJO is usually seen in both convection and winds.

OLR is often used as a proxy for tropical deep convection.

Lower OLR usually indicates enhanced deep convection.

U850 captures lower-tropospheric zonal wind anomalies.

U200 captures upper-tropospheric zonal wind anomalies.

For the MJO, these fields are dynamically connected.

Enhanced tropical convection is often accompanied by low-level wind anomalies and upper-level wind anomalies with opposite sign.

This vertical wind reversal is part of the large-scale baroclinic structure of the MJO.

Therefore, using OLR alone may miss part of the signal.

Using OLR, U850, and U200 together can improve the signal-to-noise ratio.

## 3. EOF analysis as a spatial filter

Empirical orthogonal function analysis, or EOF analysis, finds dominant spatial patterns of variability.

In this notebook, the spatial dimension is longitude.

After averaging over 15°S–15°N, each variable becomes a longitude-time field:

OLR(time, lon)  
U850(time, lon)  
U200(time, lon)

We will combine these fields into one multivariate matrix.

EOF analysis then finds patterns that describe coherent variability across all three variables and all longitudes.

If the leading EOF pair resembles the large-scale MJO structure, then projecting daily data onto those EOFs gives principal components that act like an MJO index.

In this sense, EOF projection is not just a dimension-reduction method.

It also acts as a spatial filter.

## 4. Why spectral analysis is still important

EOF analysis tells us which spatial patterns dominate.

But EOF analysis alone does not prove that those patterns represent the MJO.

To test whether the leading PCs are MJO-like, we need to examine their time scales.

This is where spectral analysis enters.

If PC1 and PC2 represent the MJO, their spectra should show enhanced variance at intraseasonal periods.

In Wheeler and Hendon (2004), the key period band was approximately 30–80 days.

In this notebook, we will examine both:

- 30–80 days, following Wheeler and Hendon (2004)
- 30–90 days, a common broader MJO band

The main question is:

> After multivariate EOF projection, do PC1 and PC2 concentrate variance in the MJO time-scale band?

## 5. What this notebook will and will not do

This notebook will build a simplified RMM-like index.

It will show how multivariate EOF analysis can extract MJO-like principal components from daily OLR, U850, and U200.

It will also use spectral analysis to evaluate whether those PCs emphasize intraseasonal variability.

However, this notebook will not reproduce every detail of the operational RMM index.

For example, we will keep the preprocessing relatively simple and focus on the statistical logic:

combined fields  
→ EOF patterns  
→ PC time series  
→ spectral evidence for MJO time scales

The goal is not to create an operational monitoring product.

The goal is to understand why multivariate EOFs can reveal a clearer MJO signal than simple area-mean spectral analysis.

## 6. Key idea of Part I

The main idea of this part is:

> A raw area-mean spectrum asks whether a fixed regional average contains MJO-period variability.

A multivariate EOF-based spectrum asks a different question:

> After extracting the dominant large-scale coupled convection-circulation pattern, does the resulting index vary mainly on MJO time scales?

The second question is closer to the logic of Wheeler and Hendon (2004).

That is why the spectral signal of the EOF PCs can be much clearer than the spectral signal of raw area-mean OLR.

# Part II. Data and minimal preprocessing

## 7. Import packages

We first import the packages used in this notebook.

The data preparation here is brief because the previous notebook already introduced the main preprocessing steps in detail.

In [ ]:
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import s3fs

from scipy import signal
from scipy import stats

## 8. Define basic settings

We use daily data from 1979 to 2024.

Following Wheeler and Hendon (2004), we focus on near-equatorial fields.

Here we use:

$$
15^\circ S - 15^\circ N
$$

for the latitude mean.

The key MJO spectral bands used later are:

- 30–80 days, following Wheeler and Hendon (2004)
- 30–90 days, a common broader MJO band

In [ ]:
# Analysis period
start_date = "1979-01-01"
end_date = "2024-12-31"

# Near-equatorial latitude band
lat_bounds = (-15, 15)

# Daily sampling interval
dt = 1.0

# Low-frequency removal window
lowfreq_window = 120

# MJO bands
mjo_band_wh04 = (30, 80)
mjo_band_broad = (30, 90)

## 9. Read OLR, U850, and U200

We use the same Pythia cloud Zarr datasets as in the previous notebook.

The variables are:

- OLR from NOAA
- U850 from NCEP-NCAR reanalysis
- U200 from NCEP-NCAR reanalysis

For the wind data, we use `drop=True` when selecting the pressure level so that the extra `level` coordinate does not cause alignment problems later.

In [ ]:
URL = "https://js2.jetstream-cloud.org:8001/"
fs = s3fs.S3FileSystem(anon=True, client_kwargs=dict(endpoint_url=URL))
# OLR
olr_noaa_store = s3fs.S3Map(
    root="pythia/olr_noaa.zarr",
    s3=fs,
    check=False
)

olr = xr.open_zarr(olr_noaa_store)
olr = olr.rename_vars({"__xarray_dataarray_variable__": "olr"})

# SST is not used directly in this simplified notebook,
# but this is the same dataset used in the previous notebook.
sst_noaa_oi_store = s3fs.S3Map(
    root="pythia/sst_noaa_oi.zarr",
    s3=fs,
    check=False
)

sst = xr.open_zarr(sst_noaa_oi_store)
sst = sst.rename_vars({"__xarray_dataarray_variable__": "sst"})
# Zonal wind
uwind_ncep_ncar_store = s3fs.S3Map(
    root="pythia/uwind-ncep-ncar.zarr",
    s3=fs,
    check=False
)

uwind_ncep_ncar = xr.open_zarr(uwind_ncep_ncar_store)

# Select U850 and U200.
# The use of drop=True removes the selected pressure-level coordinate.
u850 = uwind_ncep_ncar["uwnd"].isel(level=0, drop=True).to_dataset(name="u850")
u200 = uwind_ncep_ncar["uwnd"].isel(level=1, drop=True).to_dataset(name="u200")

## 10. Standardize time and longitude coordinates

The OLR and wind datasets may use slightly different daily time stamps.

For example, one dataset may use 12:00 UTC and another may use 00:00 UTC.

For daily analysis, we convert all time coordinates to daily dates.

We also make sure longitude uses the 0–360 convention.

In [ ]:
def convert_lon_to_360(da):
    """
    Convert longitude coordinate to 0-360 convention if needed.
    """

    lon = da["lon"]

    if lon.min() < 0:
        da = da.assign_coords(lon=(lon % 360))
        da = da.sortby("lon")

    return da
def standardize_time_and_lon(da):
    """
    Standardize daily time stamps and longitude convention.
    """

    da = convert_lon_to_360(da)

    da = da.assign_coords(
        time=da["time"].dt.floor("D")
    )

    return da

In [ ]:
olr_da = standardize_time_and_lon(olr["olr"])
u850_da = standardize_time_and_lon(u850["u850"])
u200_da = standardize_time_and_lon(u200["u200"])

olr_da = olr_da.sel(time=slice(start_date, end_date))
u850_da = u850_da.sel(time=slice(start_date, end_date))
u200_da = u200_da.sel(time=slice(start_date, end_date))

## 11. Compute 15°S–15°N latitude means

Next, we average each field over 15°S–15°N while keeping longitude.

After this step, each variable has dimensions:

```text
time, lon

In [ ]:
def lat_mean_robust(da, lat_bounds):
    """
    Compute cosine-latitude-weighted latitude mean while keeping longitude.

    This function handles both increasing and decreasing latitude coordinates.
    """

    lat1, lat2 = lat_bounds

    lat_values = da["lat"].values

    if lat_values[0] < lat_values[-1]:
        lat_slice = slice(lat1, lat2)
    else:
        lat_slice = slice(lat2, lat1)

    da_reg = da.sel(lat=lat_slice)

    weights = np.cos(np.deg2rad(da_reg["lat"]))

    da_mean = da_reg.weighted(weights).mean(dim="lat")

    return da_mean

In [ ]:
olr_eq = lat_mean_robust(olr_da, lat_bounds=lat_bounds).load()
u850_eq = lat_mean_robust(u850_da, lat_bounds=lat_bounds).load()
u200_eq = lat_mean_robust(u200_da, lat_bounds=lat_bounds).load()

## 12. Fill missing values

Before removing the seasonal cycle and applying EOF analysis, we fill missing values using linear interpolation along time.

This step is mainly needed for OLR.

In [ ]:
def fill_missing_time(da):
    """
    Fill missing values by linear interpolation along time.
    """

    n_nan_before = int(da.isnull().sum().values)

    da_filled = da.interpolate_na(
        dim="time",
        method="linear"
    )

    n_nan_after = int(da_filled.isnull().sum().values)

    print(f"Missing values before interpolation: {n_nan_before}")
    print(f"Missing values after interpolation:  {n_nan_after}")

    if n_nan_after != 0:
        raise ValueError(
            "There are still missing values after interpolation. "
            "Check whether missing values occur at the beginning or end of the time series."
        )

    return da_filled

In [ ]:
olr_eq = fill_missing_time(olr_eq)
u850_eq = fill_missing_time(u850_eq)
u200_eq = fill_missing_time(u200_eq)

## 13. Remove the mean seasonal cycle

We remove the daily climatological seasonal cycle at each longitude.

This gives seasonal-cycle-removed anomalies.

This step was explained in detail in the previous notebook, so here we only apply it.

In [ ]:
def remove_leap_days(da):
    """
    Remove February 29 so that every year has 365 days.
    """

    is_feb29 = (da["time"].dt.month == 2) & (da["time"].dt.day == 29)

    da_no_leap = da.sel(time=~is_feb29)

    return da_no_leap
def remove_daily_climatology(da):
    """
    Remove daily climatology from a DataArray.
    """

    da_no_leap = remove_leap_days(da)

    clim = da_no_leap.groupby("time.dayofyear").mean("time")

    anom = da_no_leap.groupby("time.dayofyear") - clim

    return anom, clim

In [ ]:
olr_anom, olr_clim = remove_daily_climatology(olr_eq)
u850_anom, u850_clim = remove_daily_climatology(u850_eq)
u200_anom, u200_clim = remove_daily_climatology(u200_eq)

## 14. Remove low-frequency variability

Wheeler and Hendon (2004) removed longer-time-scale variability before EOF analysis.

In this simplified notebook, we remove low-frequency variability by subtracting a 120-day running mean.

This is not a full reproduction of the operational RMM preprocessing, but it follows the same basic idea:

> remove variability slower than the MJO time scale before extracting the MJO structure.

In [ ]:
def remove_low_frequency(da, window=120):
    """
    Remove low-frequency variability by subtracting a running mean.

    The rolling mean uses the current and previous time steps.
    """

    low_frequency = da.rolling(
        time=window,
        min_periods=window
    ).mean()

    high_frequency = da - low_frequency

    high_frequency = high_frequency.dropna(dim="time")

    return high_frequency

In [ ]:
olr_prime = remove_low_frequency(
    olr_anom,
    window=lowfreq_window
)

u850_prime = remove_low_frequency(
    u850_anom,
    window=lowfreq_window
)

u200_prime = remove_low_frequency(
    u200_anom,
    window=lowfreq_window
)

## 15. Align the three fields

The three variables must have exactly the same time and longitude coordinates before being combined.

We align them using an inner join.

In [ ]:
olr_prime, u850_prime, u200_prime = xr.align(
    olr_prime,
    u850_prime,
    u200_prime,
    join="inner"
)

In [ ]:
print(olr_prime)
print(u850_prime)
print(u200_prime)

In [ ]:
print("OLR shape: ", olr_prime.shape)
print("U850 shape:", u850_prime.shape)
print("U200 shape:", u200_prime.shape)

print("OLR time range: ", str(olr_prime["time"].values[0]), "to", str(olr_prime["time"].values[-1]))
print("U850 time range:", str(u850_prime["time"].values[0]), "to", str(u850_prime["time"].values[-1]))
print("U200 time range:", str(u200_prime["time"].values[0]), "to", str(u200_prime["time"].values[-1]))

At this stage, the three fields are ready for multivariate EOF analysis.

Each field has dimensions:

```text
time, lon

# Part III. Build the multivariate EOF input matrix

## 16. Why normalize the variables?

OLR, U850, and U200 have different physical units and different variances.

If we combine them directly, the EOF analysis may be dominated by whichever variable has the largest numerical variance.

To avoid this, we normalize each variable by a single global standard deviation.

Here, "global" means all times and all longitudes in the equatorial latitude-mean field.

This preserves the longitude dependence of variance within each variable, but places OLR, U850, and U200 on comparable scales.

In [ ]:
def normalize_field_global(da):
    """
    Normalize a longitude-time field by one global standard deviation.

    The mean is also removed for numerical consistency.
    """

    field_mean = da.mean(dim=("time", "lon"))
    field_std = da.std(dim=("time", "lon"))

    da_norm = (da - field_mean) / field_std

    return da_norm, field_mean, field_std

In [ ]:
olr_norm, olr_mean, olr_std = normalize_field_global(olr_prime)
u850_norm, u850_mean, u850_std = normalize_field_global(u850_prime)
u200_norm, u200_mean, u200_std = normalize_field_global(u200_prime)

In [ ]:
print(f"OLR global standard deviation:  {float(olr_std.values):.3f}")
print(f"U850 global standard deviation: {float(u850_std.values):.3f}")
print(f"U200 global standard deviation: {float(u200_std.values):.3f}")

## 17. Convert each field to a time-by-longitude matrix

EOF analysis requires a two-dimensional matrix.

The rows are time samples.

The columns are spatial features.

For each variable, the spatial feature is longitude.

In [ ]:
# Make sure all fields are ordered as time, lon
olr_norm = olr_norm.transpose("time", "lon")
u850_norm = u850_norm.transpose("time", "lon")
u200_norm = u200_norm.transpose("time", "lon")

olr_matrix = olr_norm.values
u850_matrix = u850_norm.values
u200_matrix = u200_norm.values

## 18. Combine OLR, U850, and U200 into one matrix

Now we concatenate the three matrices along the feature dimension.

The combined matrix has the form:

```text
X = [OLR(lon), U850(lon), U200(lon)]
It's dimensions are: time × combined_features
combined_features = 3 × number_of_longitudes

In [ ]:
X = np.concatenate(
    [olr_matrix, u850_matrix, u200_matrix],
    axis=1
)

# Remove any small residual column means
X = X - np.mean(X, axis=0, keepdims=True)

In [ ]:
n_time = X.shape[0]
n_features = X.shape[1]
n_lon = olr_norm.sizes["lon"]

print(f"Number of time samples: {n_time}")
print(f"Number of longitudes:   {n_lon}")
print(f"Number of features:     {n_features}")

## 19. Save metadata for later interpretation

After EOF analysis, each EOF will be one long vector.

We need to split that vector back into three longitude-dependent parts:

- OLR part
- U850 part
- U200 part

The following slices keep track of where each variable is stored in the combined matrix.

In [ ]:
field_slices = {
    "OLR": slice(0, n_lon),
    "U850": slice(n_lon, 2 * n_lon),
    "U200": slice(2 * n_lon, 3 * n_lon),
}

lon = olr_norm["lon"].values
time = olr_norm["time"].values

In [ ]:
print(field_slices)

## 20. Final checks before EOF analysis

Before computing EOFs, we check that the combined matrix does not contain missing values or infinite values.

In [ ]:
print("Any NaN in X: ", np.isnan(X).any())
print("Any Inf in X: ", np.isinf(X).any())

In [ ]:
if np.isnan(X).any():
    raise ValueError("The combined EOF matrix contains NaN values.")

if np.isinf(X).any():
    raise ValueError("The combined EOF matrix contains infinite values.")

The combined matrix is now ready for EOF analysis.

The next part will compute EOFs and PCs from this matrix.

The leading EOF pair will then be examined as a simple RMM-like representation of the MJO.

# Part IV. Multivariate EOF analysis

## 21. Compute EOFs using singular value decomposition

We now compute EOFs of the combined matrix:

X = [OLR(lon), U850(lon), U200(lon)]

The rows of X are time samples.

The columns of X are combined longitude-variable features.

We use singular value decomposition, or SVD:

X = U S Vᵀ

The EOFs are the rows of Vᵀ.

The PCs are given by U S.

This is a direct way to compute EOFs from a centered data matrix.

In [ ]:
# Convert to float32 to reduce memory use
X_eof = X.astype(np.float32)

# Number of samples
n_time = X_eof.shape[0]

# Compute feature covariance matrix
# Shape: n_features x n_features
C = (X_eof.T @ X_eof) / (n_time - 1)

# Eigen decomposition of the covariance matrix
eigvals, eigvecs = np.linalg.eigh(C)

# Sort from largest to smallest eigenvalue
idx = np.argsort(eigvals)[::-1]

eigvals = eigvals[idx]
eigvecs = eigvecs[:, idx]

# EOFs as rows: mode x feature
eofs = eigvecs.T

# PCs: time x mode
pcs = X_eof @ eigvecs

# Explained variance
eigenvalues = eigvals
variance_fraction = eigenvalues / np.sum(eigenvalues)

## 22. Examine explained variance

The first check is how much variance is explained by the leading EOF modes.

In Wheeler and Hendon (2004), the leading two EOFs of the combined OLR, U850, and U200 fields form the basis of the RMM index.

Here we check whether the leading pair also stands out in this simplified workflow.

In [ ]:
n_modes_show = 10

for i in range(n_modes_show):
    print(
        f"EOF{i+1}: explained variance = "
        f"{variance_fraction[i] * 100:.2f}%"
    )

print("")
print(
    "EOF1 + EOF2 explained variance = "
    f"{np.sum(variance_fraction[:2]) * 100:.2f}%"
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

mode_numbers = np.arange(1, n_modes_show + 1)

ax.bar(
    mode_numbers,
    variance_fraction[:n_modes_show] * 100
)

ax.set_xlabel("EOF mode")
ax.set_ylabel("Explained variance [%]")
ax.set_title("Explained variance of leading EOF modes")
ax.set_xticks(mode_numbers)

plt.show()

The leading EOFs do not need to explain most of the total variance.

The input matrix contains all longitudes, three variables, all seasons, and many time scales.

Therefore, even a modest explained variance can still represent a physically meaningful large-scale mode.

The key question is not only how much variance EOF1 and EOF2 explain, but also whether their PCs vary mainly on MJO time scales.

## 23. Normalize the PCs

EOF signs and amplitudes are arbitrary up to a scaling convention.

For plotting and spectral analysis, it is convenient to use standardized PCs:

$$
PC_{norm} = \frac{PC - \overline{PC}}{\sigma_{PC}}
$$

The EOF patterns are rescaled consistently so that the product of normalized PCs and rescaled EOFs still represents the same variability.

In [ ]:
# Standardize PCs
pc_mean = np.mean(pcs, axis=0, keepdims=True)
pc_std = np.std(pcs, axis=0, ddof=1, keepdims=True)

pcs_norm = (pcs - pc_mean) / pc_std

# Rescale EOF patterns consistently
# This means: X ≈ pcs_norm @ eof_patterns
eof_patterns = eofs * pc_std.T

EOF signs are arbitrary.

Multiplying an EOF by -1 and multiplying its PC by -1 gives the same reconstruction.

For easier visual comparison, we apply a simple sign convention to the first two modes.

This sign convention does not affect the spectrum, explained variance, or physical interpretation.

In [ ]:
def apply_simple_sign_convention(eof_patterns, pcs_norm, field_slices, lon):
    """
    Apply a simple sign convention to EOF1 and EOF2.

    If the mean OLR loading over 80E-150E is positive,
    multiply both the EOF pattern and PC by -1.

    This is only for plotting convenience.
    """

    eof_patterns_out = eof_patterns.copy()
    pcs_norm_out = pcs_norm.copy()

    lon_mask = (lon >= 80) & (lon <= 150)

    for mode in [0, 1]:
        olr_part = eof_patterns_out[mode, field_slices["OLR"]]

        if np.nanmean(olr_part[lon_mask]) > 0:
            eof_patterns_out[mode, :] *= -1
            pcs_norm_out[:, mode] *= -1

    return eof_patterns_out, pcs_norm_out

In [ ]:
eof_patterns_plot, pcs_norm_plot = apply_simple_sign_convention(
    eof_patterns=eof_patterns,
    pcs_norm=pcs_norm,
    field_slices=field_slices,
    lon=lon
)

## 24. Split EOF patterns back into OLR, U850, and U200 parts

Each EOF is one long vector with length:

$$
3 \times N_{lon}
$$

We split each EOF into three longitude-dependent parts:

- OLR component
- U850 component
- U200 component

This allows us to plot the spatial structure of each EOF as a function of longitude.

In [ ]:
def split_eof_pattern(eof_vector, field_slices):
    """
    Split one combined EOF vector into OLR, U850, and U200 parts.
    """

    eof_olr = eof_vector[field_slices["OLR"]]
    eof_u850 = eof_vector[field_slices["U850"]]
    eof_u200 = eof_vector[field_slices["U200"]]

    return eof_olr, eof_u850, eof_u200

In [ ]:
eof1_olr, eof1_u850, eof1_u200 = split_eof_pattern(
    eof_patterns_plot[0, :],
    field_slices
)

eof2_olr, eof2_u850, eof2_u200 = split_eof_pattern(
    eof_patterns_plot[1, :],
    field_slices
)

eof3_olr, eof3_u850, eof3_u200 = split_eof_pattern(
    eof_patterns_plot[2, :],
    field_slices
)

## 25. Plot EOF1 and EOF2 spatial structures

We now plot the longitude structure of EOF1 and EOF2.

These patterns show how OLR, U850, and U200 vary together in the leading modes.

If the leading pair is MJO-like, OLR and wind should show a large-scale coupled convection-circulation structure across the tropics.

The exact sign is not important because EOF signs are arbitrary.

In [ ]:
fig, axes = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(11, 7),
    sharex=True
)

# EOF1
axes[0].plot(lon, eof1_olr, linewidth=2, label="OLR")
axes[0].plot(lon, eof1_u850, linewidth=2, label="U850")
axes[0].plot(lon, eof1_u200, linewidth=2, label="U200")
axes[0].axhline(0, linewidth=0.8)
axes[0].set_title(
    f"EOF1 structure, explained variance = {variance_fraction[0] * 100:.2f}%"
)
axes[0].set_ylabel("Rescaled EOF loading")
axes[0].legend()

# EOF2
axes[1].plot(lon, eof2_olr, linewidth=2, label="OLR")
axes[1].plot(lon, eof2_u850, linewidth=2, label="U850")
axes[1].plot(lon, eof2_u200, linewidth=2, label="U200")
axes[1].axhline(0, linewidth=0.8)
axes[1].set_title(
    f"EOF2 structure, explained variance = {variance_fraction[1] * 100:.2f}%"
)
axes[1].set_xlabel("Longitude [degrees east]")
axes[1].set_ylabel("Rescaled EOF loading")
axes[1].legend()

plt.tight_layout()
plt.show()

EOF2: enhanced convection more over the Indian Ocean

EOF1: enhanced convection shifted eastward toward the Maritime Continent

In Wheeler and Hendon (2004), EOF1 and EOF2 form a pair.

They are not interpreted as two unrelated modes.

Together, they describe different phases of a propagating MJO-like structure.

## 26. Plot PC1 and PC2 time series

The principal components describe how strongly each EOF pattern appears on each day.

If EOF1 and EOF2 form an MJO-like pair, PC1 and PC2 should vary mainly on intraseasonal time scales.

We first plot a short example period.

In [ ]:
pc_da = xr.Dataset(
    {
        "PC1": ("time", pcs_norm_plot[:, 0]),
        "PC2": ("time", pcs_norm_plot[:, 1]),
        "PC3": ("time", pcs_norm_plot[:, 2]),
    },
    coords={
        "time": time
    }
)

pc_da

In [ ]:
pc_plot_start = "2001-01-01"
pc_plot_end = "2002-12-31"

fig, ax = plt.subplots(figsize=(13, 4))

pc_da["PC1"].sel(time=slice(pc_plot_start, pc_plot_end)).plot(
    ax=ax,
    linewidth=1.2,
    label="PC1"
)

pc_da["PC2"].sel(time=slice(pc_plot_start, pc_plot_end)).plot(
    ax=ax,
    linewidth=1.2,
    label="PC2"
)

ax.axhline(0, linewidth=0.8)
ax.set_title("Standardized PC1 and PC2")
ax.set_xlabel("Time")
ax.set_ylabel("Standardized PC")
ax.legend()

plt.show()

## 27. Quick interpretation of the leading EOF pair

EOF1 and EOF2 should be interpreted together.

A propagating oscillation is often represented by a pair of EOFs in approximate quadrature.

That means EOF1 and EOF2 describe similar large-scale structures shifted in longitude and phase.

The PCs should also show a phase relationship in time.

However, visual inspection is not enough.

In the next part, we use spectral analysis to test whether PC1 and PC2 concentrate variance in the MJO period band.

# Part V. Spectral analysis of EOF principal components

## 28. Why analyze the spectra of PCs?

We now examine the spectra of the EOF principal components.

This follows the logic of Wheeler and Hendon (2004).

The key question is:

> After projecting OLR, U850, and U200 onto the leading multivariate EOFs, do PC1 and PC2 concentrate variance in the MJO time-scale band?

Following Wheeler and Hendon (2004), we focus on the 30–80 day period band.

If PC1 and PC2 represent MJO-like variability, their spectra should show enhanced variance in this intraseasonal band.

PC3 is included as a comparison mode.

In [ ]:
# MJO band used by Wheeler and Hendon (2004)
mjo_band = (30, 80)

## 29. Compute area-conserving spectra

Wheeler and Hendon (2004) plotted spectra in an area-conserving format.

In this format, the x-axis is logarithmic frequency and the plotted quantity is:

$$
Power \times Frequency
$$

With this convention, the area under the curve within a frequency band is proportional to the variance in that band.

This makes it easier to visually compare how much variance lies in the 30–80 day MJO band.

In [ ]:
def smooth_121(y, passes=20):
    """
    Smooth a one-dimensional spectrum using repeated 1-2-1 smoothing.
    """

    y_smooth = np.asarray(y, dtype=float).copy()

    for _ in range(passes):
        y_new = y_smooth.copy()

        y_new[1:-1] = (
            0.25 * y_smooth[:-2]
            + 0.50 * y_smooth[1:-1]
            + 0.25 * y_smooth[2:]
        )

        y_smooth = y_new

    return y_smooth

def compute_pc_spectrum_wh04_style(x, dt=1.0, smooth_passes=20):
    """
    Compute a Wheeler-Hendon-style PC spectrum.

    The returned display quantity is power multiplied by frequency.
    """

    x = np.asarray(x)
    x = x - np.mean(x)

    fs = 1.0 / dt

    freq, psd = signal.periodogram(
        x,
        fs=fs,
        window="boxcar",
        detrend=False,
        scaling="density"
    )

    # Remove zero frequency
    freq = freq[1:]
    psd = psd[1:]

    # Smooth spectrum
    psd_smooth = smooth_121(
        psd,
        passes=smooth_passes
    )

    period = 1.0 / freq

    # Area-conserving display quantity
    power_times_frequency = psd_smooth * freq

    return {
        "frequency": freq,
        "period": period,
        "psd": psd,
        "psd_smooth": psd_smooth,
        "power_times_frequency": power_times_frequency,
    }

In [ ]:
pc1_spec = compute_pc_spectrum_wh04_style(
    pc_da["PC1"].values,
    dt=dt,
    smooth_passes=20
)

pc2_spec = compute_pc_spectrum_wh04_style(
    pc_da["PC2"].values,
    dt=dt,
    smooth_passes=20
)

pc3_spec = compute_pc_spectrum_wh04_style(
    pc_da["PC3"].values,
    dt=dt,
    smooth_passes=20
)

## 30. Compute red-noise reference spectra

Wheeler and Hendon (2004) compared the PC spectra with red-noise spectra estimated from lag-1 autocorrelation.

Here we use the same idea.

The red-noise curve is not used here as a formal significance test.

It is a reference spectrum showing how much intraseasonal variance would be expected from a simple persistent red-noise process.

In [ ]:
def lag1_autocorrelation(x):
    """
    Compute lag-1 autocorrelation of a one-dimensional time series.
    """

    x = np.asarray(x)
    x = x[np.isfinite(x)]
    x = x - np.mean(x)

    return np.corrcoef(x[:-1], x[1:])[0, 1]
def red_noise_reference_spectrum(freq, x, dt=1.0, smooth_passes=20):
    """
    Compute a scaled AR(1) red-noise reference spectrum.

    The shape is based on lag-1 autocorrelation.
    The spectrum is scaled to match the variance of the input time series.
    """

    x = np.asarray(x)
    x = x - np.mean(x)

    alpha = lag1_autocorrelation(x)
    variance = np.var(x)

    red_shape = (
        (1 - alpha ** 2)
        / (1 + alpha ** 2 - 2 * alpha * np.cos(2 * np.pi * freq * dt))
    )

    # Scale red-noise spectrum to match total variance
    red_psd = red_shape / np.trapezoid(red_shape, freq) * variance

    red_psd_smooth = smooth_121(
        red_psd,
        passes=smooth_passes
    )

    return {
        "alpha": alpha,
        "red_psd": red_psd_smooth,
        "red_power_times_frequency": red_psd_smooth * freq,
    }

In [ ]:
pc1_red = red_noise_reference_spectrum(
    pc1_spec["frequency"],
    pc_da["PC1"].values,
    dt=dt,
    smooth_passes=20
)

pc2_red = red_noise_reference_spectrum(
    pc2_spec["frequency"],
    pc_da["PC2"].values,
    dt=dt,
    smooth_passes=20
)

pc3_red = red_noise_reference_spectrum(
    pc3_spec["frequency"],
    pc_da["PC3"].values,
    dt=dt,
    smooth_passes=20
)

## 31. Compute 30–80 day variance fractions

We now compute the fraction of total spectral variance that lies in the 30–80 day band.

This is the key number reported by Wheeler and Hendon (2004).

For an MJO-like leading EOF pair, PC1 and PC2 should have relatively large 30–80 day variance fractions.

PC3 should usually be less dominated by this band.

In [ ]:
def spectral_band_fraction(freq, psd, band=(30, 80)):
    """
    Compute the fraction of spectral variance within a period band.

    Parameters
    ----------
    freq : array-like
        Frequency in cycles per day.
    psd : array-like
        Power spectral density.
    band : tuple
        Period band in days.

    Returns
    -------
    frac : float
        Fraction of total spectral variance within the band.
    """

    freq = np.asarray(freq)
    psd = np.asarray(psd)

    period = 1.0 / freq

    band_min, band_max = band

    band_mask = (period >= band_min) & (period <= band_max)

    total_power = np.trapezoid(psd, freq)
    band_power = np.trapezoid(psd[band_mask], freq[band_mask])

    frac = band_power / total_power

    return frac

In [ ]:
pc_specs = {
    "PC1": pc1_spec,
    "PC2": pc2_spec,
    "PC3": pc3_spec,
}

red_specs = {
    "PC1 red noise": {
        "frequency": pc1_spec["frequency"],
        "psd": pc1_red["red_psd"],
    },
    "PC2 red noise": {
        "frequency": pc2_spec["frequency"],
        "psd": pc2_red["red_psd"],
    },
    "PC3 red noise": {
        "frequency": pc3_spec["frequency"],
        "psd": pc3_red["red_psd"],
    },
}

In [ ]:
pc_band_fraction_results = {}

for name, spec in pc_specs.items():
    frac = spectral_band_fraction(
        spec["frequency"],
        spec["psd_smooth"],
        band=mjo_band
    )

    pc_band_fraction_results[name] = frac

    print(f"{name}: 30-80 day variance fraction = {frac:.3f}")

In [ ]:
red_band_fraction_results = {}

for name, spec in red_specs.items():
    frac = spectral_band_fraction(
        spec["frequency"],
        spec["psd"],
        band=mjo_band
    )

    red_band_fraction_results[name] = frac

    print(f"{name}: 30-80 day variance fraction = {frac:.3f}")

In [ ]:
band_fraction_df = pd.DataFrame(
    {
        "PC spectrum": pc_band_fraction_results,
        "Red-noise reference": {
            "PC1": red_band_fraction_results["PC1 red noise"],
            "PC2": red_band_fraction_results["PC2 red noise"],
            "PC3": red_band_fraction_results["PC3 red noise"],
        },
    }
)

band_fraction_df

## 32. Plot PC spectra in Wheeler-Hendon style

We now plot PC1, PC2, and PC3 spectra.

The solid curve is the PC spectrum.

The dashed curve is the red-noise reference.

The vertical dotted lines mark the 30–80 day band.

This is the closest plot in this notebook to the spectral analysis in Wheeler and Hendon (2004).

In [ ]:
fig, axes = plt.subplots(
    nrows=3,
    ncols=1,
    figsize=(9, 9),
    sharex=True
)

spec_list = [pc1_spec, pc2_spec, pc3_spec]
red_list = [pc1_red, pc2_red, pc3_red]
name_list = ["PC1", "PC2", "PC3"]
expvar_list = variance_fraction[:3] * 100

for ax, spec, red, name, expvar in zip(
    axes,
    spec_list,
    red_list,
    name_list,
    expvar_list
):
    frac = pc_band_fraction_results[name]

    ax.plot(
        spec["frequency"],
        spec["power_times_frequency"],
        linewidth=2,
        label=name
    )

    ax.plot(
        spec["frequency"],
        red["red_power_times_frequency"],
        linestyle="--",
        linewidth=1.5,
        label="Red-noise reference"
    )

    # 30-80 day band
    ax.axvline(1 / 80, linestyle=":", linewidth=1)
    ax.axvline(1 / 30, linestyle=":", linewidth=1)

    ax.set_xscale("log")
    ax.set_ylabel("Power × frequency")

    ax.set_title(
        f"{name}, ExpVar = {expvar:.2f}%, "
        f"Var in 30–80 d band = {frac:.2f}"
    )

    ax.legend()

axes[-1].set_xlabel("Frequency [cycles per day]")

plt.tight_layout()
plt.show()

The 30–80 day band is marked by the two vertical dotted lines.

If the leading EOF pair is MJO-like, PC1 and PC2 should show clear enhancement within this band.

The red-noise curve provides a simple background reference.

A useful result would be:

- PC1 and PC2 have larger 30–80 day variance fractions than their red-noise references.
- PC1 and PC2 have larger 30–80 day variance fractions than PC3.

That would support the interpretation that the leading EOF pair isolates MJO-like intraseasonal variability.

## 33. Plot 30–80 day variance fractions

The spectra give a visual diagnosis.

The variance fraction gives a compact numerical summary.

Here we compare the 30–80 day variance fraction for PC1, PC2, and PC3 against their red-noise references.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

modes = ["PC1", "PC2", "PC3"]
x = np.arange(len(modes))
width = 0.35

pc_values = [pc_band_fraction_results[m] for m in modes]

red_values = [
    red_band_fraction_results[f"{m} red noise"]
    for m in modes
]

ax.bar(
    x - width / 2,
    pc_values,
    width,
    label="PC spectrum"
)

ax.bar(
    x + width / 2,
    red_values,
    width,
    label="Red-noise reference"
)

ax.set_xticks(x)
ax.set_xticklabels(modes)
ax.set_ylabel("30–80 day variance fraction")
ax.set_title("Variance fraction in the 30–80 day band")
ax.legend()

plt.show()

## 34. Interpretation of the PC spectra

The key result is whether PC1 and PC2 concentrate variance in the 30–80 day band.

Following Wheeler and Hendon (2004), this provides evidence that the leading EOF pair captures MJO-like intraseasonal variability.

The interpretation is:

- PC1 and PC2 are the main candidates for the MJO-related pair.
- PC3 is used as a comparison mode.
- A strong 30–80 day peak in PC1 and PC2 suggests that the EOF projection has isolated intraseasonal MJO-like variability.
- A weaker or less organized PC3 spectrum suggests that PC3 is not part of the leading MJO pair.

This explains why the spectra of EOF PCs can be clearer than spectra of raw area-mean OLR.

The spatial projection has already selected a coherent convection-circulation pattern before spectral analysis is applied.

## 35. Key takeaways from Part V

In this part, we followed the spectral-analysis logic of Wheeler and Hendon (2004).

The main points are:

1. We analyzed spectra of EOF principal components, not raw area-mean OLR.

2. We focused on the 30–80 day band.

3. We plotted power multiplied by frequency, following the area-conserving style used in Wheeler and Hendon (2004).

4. We compared PC1, PC2, and PC3.

5. We used a lag-1 red-noise spectrum as a background reference.

6. If PC1 and PC2 have enhanced 30–80 day variance, this supports their interpretation as an MJO-like EOF pair.

7. If PC3 has weaker 30–80 day concentration, this supports the idea that the leading two modes are special.

The next step is to test whether PC1 and PC2 behave like a propagating pair using lag correlation, coherence, phase, and phase-space diagrams.

# Part VI. PC1–PC2 relationship and propagation diagnostics

## 36. Why examine the relationship between PC1 and PC2?

The spectra showed whether each PC has enhanced variance in the 30–80 day MJO band.

However, the MJO is not just an intraseasonal oscillation.

It is also a propagating disturbance.

A propagating mode is often represented by a pair of EOFs.

Therefore, PC1 and PC2 should not only have MJO-band power.

They should also have a consistent phase relationship.

In Wheeler and Hendon (2004), PC1 and PC2 are coherent in the 30–80 day band, and their phase relationship is close to a quarter cycle.

That is consistent with eastward propagation.

## 37. Lag correlation between PC1 and PC2

We first compute lag correlations.

Here the convention is:

> Positive lag means PC2 lags PC1.

If PC1 and PC2 form a propagating pair, the maximum correlation should occur at a nonzero lag.

For an MJO-like time scale, a quarter cycle corresponds roughly to:

- 7.5 days for a 30-day oscillation
- 15 days for a 60-day oscillation
- 20 days for an 80-day oscillation

So a peak correlation at a lag of roughly 10–15 days is physically plausible for MJO-like variability.

In [ ]:
def lead_lag_correlation(x, y, max_lag):
    """
    Compute lead-lag correlation between two time series.

    Convention:
    Positive lag means y lags x.
    """

    x = np.asarray(x)
    y = np.asarray(y)

    if len(x) != len(y):
        raise ValueError("x and y must have the same length.")

    if np.any(np.isnan(x)) or np.any(np.isnan(y)):
        raise ValueError("Input time series contains NaN values.")

    x = x - np.mean(x)
    y = y - np.mean(y)

    lags = np.arange(-max_lag, max_lag + 1)

    r = np.empty(len(lags))

    for i, lag in enumerate(lags):
        if lag < 0:
            x_lag = x[-lag:]
            y_lag = y[:lag]
        elif lag > 0:
            x_lag = x[:-lag]
            y_lag = y[lag:]
        else:
            x_lag = x
            y_lag = y

        r[i] = np.corrcoef(x_lag, y_lag)[0, 1]

    return lags, r

In [ ]:
max_lag = 60

lags, r_pc1_pc1 = lead_lag_correlation(
    pc_da["PC1"].values,
    pc_da["PC1"].values,
    max_lag=max_lag
)

lags, r_pc1_pc2 = lead_lag_correlation(
    pc_da["PC1"].values,
    pc_da["PC2"].values,
    max_lag=max_lag
)

In [ ]:
# Find maximum positive correlation between PC1 and PC2
imax = np.argmax(r_pc1_pc2)

lag_max = lags[imax]
r_max = r_pc1_pc2[imax]

print(f"Maximum PC1-PC2 correlation: r = {r_max:.3f}")
print(f"Lag of maximum correlation: {lag_max} days")
print("")
print("Lag convention:")
print("  Positive lag means PC2 lags PC1.")
print("  Negative lag means PC2 leads PC1.")

if lag_max > 0:
    print(f"Interpretation: PC2 lags PC1 by about {lag_max} days.")
elif lag_max < 0:
    print(f"Interpretation: PC2 leads PC1 by about {abs(lag_max)} days.")
else:
    print("Interpretation: PC1 and PC2 are most correlated at zero lag.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(
    lags,
    r_pc1_pc1,
    linewidth=2,
    label="PC1 with PC1"
)

ax.plot(
    lags,
    r_pc1_pc2,
    linewidth=2,
    label="PC1 with PC2"
)

ax.axhline(0, linewidth=0.8)
ax.axvline(0, linestyle="--", linewidth=1)

# Mark the lag of maximum PC1-PC2 correlation
ax.axvline(
    lag_max,
    linestyle=":",
    linewidth=1.5,
    label=f"Max PC1-PC2 r at lag = {lag_max} d"
)

ax.set_xlabel("Lag [days]")
ax.set_ylabel("Correlation")
ax.set_title("Lag correlations of PC1 and PC2")
ax.legend()

plt.show()

The lag correlation gives a simple time-domain view of the PC1–PC2 relationship.

In this example, the maximum positive correlation occurs at a negative lag.

Under our lag convention, this means that PC2 leads PC1.

This is not a problem.

EOF signs and EOF ordering are arbitrary, so the exact direction of the lead-lag relationship can depend on the chosen EOF convention.

The important result is that PC1 and PC2 show a clear phase-shifted relationship rather than behaving like unrelated time series.

This is consistent with the interpretation that EOF1 and EOF2 form a propagating paired mode.

## 38. Cross-spectrum, coherence, and phase

The lag correlation uses all time scales together.

A more frequency-specific test is cross-spectral analysis.

From the cross-spectrum, we compute:

- coherence squared
- phase

Coherence squared tells us whether PC1 and PC2 are strongly related at a given frequency.

Phase tells us whether one PC leads or lags the other.

Following Wheeler and Hendon (2004), the key diagnostic is whether PC1 and PC2 are coherent in the 30–80 day band and have a phase relationship close to a quarter cycle.

In [ ]:
def compute_cross_spectrum_wh04_style(x, y, dt=1.0, smooth_passes=20):
    """
    Compute smoothed cross-spectrum, coherence squared, and phase.

    Convention:
    Positive phase means x leads y.
    """

    x = np.asarray(x)
    y = np.asarray(y)

    x = x - np.mean(x)
    y = y - np.mean(y)

    fs = 1.0 / dt
    n = len(x)

    # Auto spectra
    freq, pxx = signal.periodogram(
        x,
        fs=fs,
        window="boxcar",
        detrend=False,
        scaling="density"
    )

    _, pyy = signal.periodogram(
        y,
        fs=fs,
        window="boxcar",
        detrend=False,
        scaling="density"
    )

    # Cross spectrum
    freq_csd, pxy = signal.csd(
        x,
        y,
        fs=fs,
        window="boxcar",
        nperseg=n,
        noverlap=0,
        detrend=False,
        scaling="density",
        return_onesided=True
    )

    if not np.allclose(freq, freq_csd):
        raise ValueError("Frequency arrays from periodogram and CSD do not match.")

    # Remove zero frequency
    freq = freq[1:]
    pxx = pxx[1:]
    pyy = pyy[1:]
    pxy = pxy[1:]

    # Smooth auto spectra
    pxx_smooth = smooth_121(pxx, passes=smooth_passes)
    pyy_smooth = smooth_121(pyy, passes=smooth_passes)

    # Smooth real and imaginary parts of cross spectrum
    pxy_real_smooth = smooth_121(np.real(pxy), passes=smooth_passes)
    pxy_imag_smooth = smooth_121(np.imag(pxy), passes=smooth_passes)

    pxy_smooth = pxy_real_smooth + 1j * pxy_imag_smooth

    coherence_squared = (
        np.abs(pxy_smooth) ** 2
        / (pxx_smooth * pyy_smooth)
    )

    # SciPy CSD uses conj(X) * Y.
    # With this definition, a negative angle means x leads y.
    # We multiply by -1 so that positive phase means x leads y.
    phase_deg = -1 * np.rad2deg(np.angle(pxy_smooth))

    period = 1.0 / freq

    return {
        "frequency": freq,
        "period": period,
        "coherence_squared": coherence_squared,
        "phase_deg": phase_deg,
        "pxy": pxy_smooth,
        "pxx": pxx_smooth,
        "pyy": pyy_smooth,
    }

In [ ]:
pc1_pc2_cross = compute_cross_spectrum_wh04_style(
    pc_da["PC1"].values,
    pc_da["PC2"].values,
    dt=dt,
    smooth_passes=20
)

pc1_pc3_cross = compute_cross_spectrum_wh04_style(
    pc_da["PC1"].values,
    pc_da["PC3"].values,
    dt=dt,
    smooth_passes=20
)

In [ ]:
def summarize_cross_spectrum_in_band(cross_spec, band=(30, 80)):
    """
    Summarize coherence squared and phase in a period band.
    """

    period = cross_spec["period"]
    coherence_squared = cross_spec["coherence_squared"]
    phase_deg = cross_spec["phase_deg"]

    band_min, band_max = band

    band_mask = (period >= band_min) & (period <= band_max)

    mean_coh2 = np.nanmean(coherence_squared[band_mask])

    phase_rad = np.deg2rad(phase_deg[band_mask])

    mean_phase_rad = stats.circmean(
        phase_rad,
        high=np.pi,
        low=-np.pi
    )

    mean_phase_deg = np.rad2deg(mean_phase_rad)

    return mean_coh2, mean_phase_deg

In [ ]:
mean_coh2_pc1_pc2, mean_phase_pc1_pc2 = summarize_cross_spectrum_in_band(
    pc1_pc2_cross,
    band=mjo_band
)

mean_coh2_pc1_pc3, mean_phase_pc1_pc3 = summarize_cross_spectrum_in_band(
    pc1_pc3_cross,
    band=mjo_band
)

print("PC1-PC2:")
print(f"  Mean coherence squared in 30-80 day band = {mean_coh2_pc1_pc2:.3f}")
print(f"  Mean phase in 30-80 day band = {mean_phase_pc1_pc2:.1f} degrees")
print("  Positive phase means PC1 leads PC2.")

print("")

print("PC1-PC3:")
print(f"  Mean coherence squared in 30-80 day band = {mean_coh2_pc1_pc3:.3f}")
print(f"  Mean phase in 30-80 day band = {mean_phase_pc1_pc3:.1f} degrees")

In [ ]:
fig, axes = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(9, 7),
    sharex=True
)

# Coherence squared
axes[0].plot(
    pc1_pc2_cross["period"],
    pc1_pc2_cross["coherence_squared"],
    linewidth=2,
    label="PC1 vs PC2"
)

axes[0].plot(
    pc1_pc3_cross["period"],
    pc1_pc3_cross["coherence_squared"],
    linewidth=2,
    label="PC1 vs PC3"
)

axes[0].axvspan(30, 80, alpha=0.2, label="30–80 day band")
axes[0].set_xscale("log")
axes[0].set_ylabel("Coherence squared")
axes[0].set_title("Cross-spectral coherence")
axes[0].set_ylim(0, 1)
axes[0].legend()

# Phase
axes[1].plot(
    pc1_pc2_cross["period"],
    pc1_pc2_cross["phase_deg"],
    linewidth=2,
    label="PC1 vs PC2"
)

axes[1].plot(
    pc1_pc3_cross["period"],
    pc1_pc3_cross["phase_deg"],
    linewidth=2,
    label="PC1 vs PC3"
)

axes[1].axhline(90, linestyle="--", linewidth=1, label="90°")
axes[1].axhline(-90, linestyle="--", linewidth=1, label="-90°")
axes[1].axhline(0, linewidth=0.8)
axes[1].axvspan(30, 80, alpha=0.2)

axes[1].set_xscale("log")
axes[1].set_xlabel("Period [days]")
axes[1].set_ylabel("Phase [degrees]")
axes[1].set_title("Cross-spectral phase")
axes[1].set_xlim(5, 365)
axes[1].invert_xaxis()
axes[1].legend()

plt.tight_layout()
plt.show()

The cross-spectral result is consistent with a propagating EOF pair.

PC1 and PC2 show high coherence squared in the 30–80 day band, meaning that the two PCs are strongly related on MJO time scales.

Their phase is close to -90 degrees in this band.

The sign of the phase depends on EOF sign, EOF ordering, and cross-spectrum convention. Therefore, the important point is not whether the phase is +90 or -90 degrees, but whether it is close to a quarter cycle.

In contrast, PC1 and PC3 have much weaker coherence in the 30–80 day band, and their phase is less stable.

This supports the interpretation that EOF1 and EOF2 form the leading MJO-like propagating pair, while EOF3 is not part of that pair.

## 39. Interpretation of coherence and phase

The coherence plot tests whether two PCs are related at a given time scale.

The phase plot tests whether one PC leads or lags the other.

For an MJO-like EOF pair, we expect:

- PC1 and PC2 to have relatively high coherence in the 30–80 day band.
- PC1 and PC2 to have a phase relationship near a quarter cycle.
- PC1 and PC3 to have weaker coherence in the same band.

A quarter cycle corresponds to about 90 degrees.

The sign of the phase can depend on EOF sign and ordering conventions.

Therefore, both +90 degrees and -90 degrees can indicate a quarter-cycle relationship, depending on how the PCs are defined.

The important point is the existence of a coherent phase-shifted relationship between PC1 and PC2 in the MJO band.

Together with the EOF structures and PC spectra, this supports the interpretation that EOF1 and EOF2 form a propagating MJO-like pair.

# Part VII. RMM-like phase space

## 40. Define a simple RMM-like index

The operational RMM index is based on two normalized principal components.

Here we define a simple RMM-like index using PC1 and PC2 from our multivariate EOF analysis.

This is not an exact reproduction of the operational RMM index.

It is a simplified teaching version.

The amplitude is defined as:

$$
A = \sqrt{PC1^2 + PC2^2}
$$

Large amplitude indicates that the leading EOF pair is strongly expressed.

Small amplitude indicates weak projection onto the leading pair.

In [ ]:
rmm_like = xr.Dataset(
    {
        "RMM1_like": pc_da["PC1"],
        "RMM2_like": pc_da["PC2"],
    }
)

rmm_like["amplitude"] = np.sqrt(
    rmm_like["RMM1_like"] ** 2
    + rmm_like["RMM2_like"] ** 2
)

rmm_like

## 41. Plot PC1-PC2 phase space for an example period

A phase-space diagram plots PC1 against PC2.

If PC1 and PC2 represent a propagating oscillation, the trajectory should tend to rotate around the origin.

Points near the origin indicate weak activity.

Points far from the origin indicate strong projection onto the leading EOF pair.

The direction of rotation can depend on EOF sign and ordering conventions, so the important feature here is organized rotation rather than the absolute direction.

In [ ]:
phase_start = "2001-01-01"
phase_end = "2002-12-31"

phase_data = rmm_like.sel(
    time=slice(phase_start, phase_end)
)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

ax.plot(
    phase_data["RMM1_like"],
    phase_data["RMM2_like"],
    linewidth=1,
    marker="o",
    markersize=2
)

# Unit circle for weak activity threshold
theta = np.linspace(0, 2 * np.pi, 300)

ax.plot(
    np.cos(theta),
    np.sin(theta),
    linestyle="--",
    linewidth=1,
    label="Amplitude = 1"
)

ax.axhline(0, linewidth=0.8)
ax.axvline(0, linewidth=0.8)

ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("PC1 / RMM1-like")
ax.set_ylabel("PC2 / RMM2-like")
ax.set_title(f"RMM-like phase space: {phase_start} to {phase_end}")
ax.legend()

plt.show()

In a phase-space diagram, organized motion around the origin indicates that PC1 and PC2 behave like a paired oscillation.

Large excursions outside the unit circle indicate stronger MJO-like activity.

Random motion near the origin indicates weak or disorganized activity.

## 42. Add simple phase categories

Wheeler and Hendon (2004) divided the RMM phase space into eight phases.

Here we define a similar simple phase category based on the angle in PC1-PC2 space.

This is useful for summarizing the state of the RMM-like index.

The phase numbers here are for teaching purposes and may not match the operational RMM phase convention exactly.

In [ ]:
def compute_rmm_like_phase(pc1, pc2):
    """
    Compute simple 8-sector phase categories from PC1 and PC2.

    Returns phase numbers from 1 to 8.
    """

    angle = np.arctan2(pc2, pc1)

    # Convert angle from [-pi, pi] to [0, 2pi]
    angle = np.mod(angle, 2 * np.pi)

    # Divide into 8 equal sectors
    phase = np.floor(angle / (2 * np.pi / 8)).astype(int) + 1

    return phase

In [ ]:
rmm_like["phase"] = xr.DataArray(
    compute_rmm_like_phase(
        rmm_like["RMM1_like"].values,
        rmm_like["RMM2_like"].values
    ),
    coords={"time": rmm_like["time"]},
    dims=["time"]
)

rmm_like

In [ ]:
phase_counts = rmm_like["phase"].to_series().value_counts().sort_index()

phase_counts

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.bar(
    phase_counts.index.astype(str),
    phase_counts.values
)

ax.set_xlabel("RMM-like phase")
ax.set_ylabel("Number of days")
ax.set_title("Number of days in each RMM-like phase")

plt.show()

The phase categories divide the PC1-PC2 plane into eight sectors.

They are useful for compositing or tracking the evolution of the index.

However, because this is a simplified RMM-like index, the phase labels should be interpreted cautiously.

The main purpose here is to show how a two-PC EOF pair can be converted into an index amplitude and phase.

## 43. Plot phase space colored by amplitude

Finally, we plot all available days in PC1-PC2 phase space and color the points by amplitude.

This shows the distribution of weak and strong events in the simplified RMM-like index.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

sc = ax.scatter(
    rmm_like["RMM1_like"],
    rmm_like["RMM2_like"],
    c=rmm_like["amplitude"],
    s=4,
    alpha=0.5
)

theta = np.linspace(0, 2 * np.pi, 300)

ax.plot(
    np.cos(theta),
    np.sin(theta),
    linestyle="--",
    linewidth=1
)

ax.axhline(0, linewidth=0.8)
ax.axvline(0, linewidth=0.8)

ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("PC1 / RMM1-like")
ax.set_ylabel("PC2 / RMM2-like")
ax.set_title("All days in RMM-like phase space")

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("Amplitude")

plt.show()

The unit circle is often used as a rough threshold for weak activity.

Days inside the circle have amplitude less than 1.

Days outside the circle have stronger projection onto the leading EOF pair.

This phase-space view connects the EOF analysis to the familiar RMM-style monitoring framework.

# Part VIII. Summary and interpretation

## 44. What did multivariate EOF add beyond the previous notebook?

The previous notebook used spectra of simple time series.

Those diagnostics showed that tropical OLR and winds contain subseasonal variability.

However, area-mean spectra are not optimized for detecting the MJO because the MJO is a propagating, spatially coherent disturbance.

This notebook added a spatial filtering step.

The multivariate EOF analysis selected coherent patterns across:

- OLR
- U850
- U200
- all longitudes

The resulting PCs are therefore not raw local time series.

They are indices of large-scale coupled convection-circulation variability.

## 45. Why are the PC spectra clearer?

The PC spectra can be clearer than raw OLR spectra because EOF projection has already selected a coherent MJO-like spatial pattern.

High-frequency weather noise may appear in OLR alone.

But it is less likely to appear simultaneously in OLR, U850, and U200 with the correct large-scale baroclinic structure.

This is why multivariate EOF analysis can improve the signal-to-noise ratio.

The key logic is:

raw fields  
→ spatial projection onto MJO-like EOFs  
→ PC time series  
→ spectral analysis of PCs

The spectrum is not applied to arbitrary area-mean OLR.

It is applied after extracting a large-scale coupled pattern.

## 46. What this simplified notebook did not include

This notebook followed the core statistical logic of Wheeler and Hendon (2004), but it did not reproduce every operational detail.

Important simplifications include:

- We used simplified preprocessing.
- We did not exactly reproduce the operational ENSO-removal procedure.
- We did not produce official RMM1 and RMM2.
- We did not compute phase composites.
- We did not perform wavenumber-frequency analysis.

These simplifications are acceptable for a teaching notebook.

The goal was to understand why multivariate EOFs can extract a clearer MJO signal than simple area-mean spectral analysis.

## 47. Final summary

This notebook built a simplified RMM-like MJO index using multivariate EOF analysis.

The main steps were:

1. Start with daily OLR, U850, and U200.

2. Average each variable over 15°S–15°N while keeping longitude.

3. Remove the seasonal cycle and low-frequency variability.

4. Normalize each variable.

5. Combine OLR, U850, and U200 into one multivariate matrix.

6. Compute EOFs of the combined field.

7. Interpret EOF1 and EOF2 as a coupled convection-circulation pair.

8. Analyze the spectra of PC1, PC2, and PC3.

9. Focus on the 30–80 day band, following Wheeler and Hendon (2004).

10. Compare the PC spectra with red-noise references.

11. Examine PC1–PC2 lag correlation, coherence, and phase.

12. Plot a simple RMM-like phase space.

The central conclusion is:

> Multivariate EOF projection acts as a spatial filter that can concentrate MJO-like variability into a small number of principal components.

This is why PC spectra can show a clearer intraseasonal MJO signal than raw area-mean OLR spectra.

## 48. Possible next notebook

A natural next step is wavenumber-frequency analysis.

The current notebook used EOFs to extract an MJO-like index.

A wavenumber-frequency notebook would instead analyze the full longitude-time field and separate:

- eastward-propagating variability
- westward-propagating variability
- standing variability

That method is useful for distinguishing the MJO from other tropical waves, such as Kelvin waves and equatorial Rossby waves.

Therefore, the next notebook can focus on:

OLR(time, lon)  
→ segmentation and tapering  
→ two-dimensional Fourier transform  
→ wavenumber-frequency spectrum  
→ eastward and westward propagation  
→ MJO spectral region

# References:
1. Wheeler, M. C., & Hendon, H. H. (2004). An all-season real-time multivariate MJO index: Development of an index for monitoring and prediction. Monthly weather review, 132(8), 1917-1932.